# Multi-view + Temporal Pretraining proj_head — Demo

This notebook demonstrates that the trained `proj_head.pt` produces meaningful embeddings:

1. **Load** the SigLIP-so400m frozen encoder + our 656K-param projection head
2. **Forward** a small batch of LIBERO frames through both
3. **Visualise** the cosine similarity matrix — same-episode pairs should be high, cross-task pairs should be low

---

## Sanity check expectations

| pair type | expected cosine sim |
|---|---|
| same agent_t (self) | 1.00 (diagonal, masked) |
| (agent_t, wrist_t) — same step, different camera | high (~0.5–0.7) |
| (agent_t, agent_{t+5}) — same camera, 5 steps later | very high (~0.7–0.9) |
| cross-episode, same task | medium |
| cross-task | low (~0.0) |

If the diagonal block (same-episode) clearly stands out from off-diagonal noise → **the projection head works**.


In [ ]:
# Setup paths
import os, sys
from pathlib import Path

REPO = Path('/home/ubuntu/ro_planning')   # change if you cloned vla-lab elsewhere
sys.path.insert(0, str(REPO / 'code'))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import torch
import numpy as np
import matplotlib.pyplot as plt
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1. Load encoder + proj_head

In [ ]:
from pretrain.train import SiglipEncoderWrapper
from pretrain.model import MultiViewProjectionHead

OPENVLA_DIR = '/path/to/openvla-7b-finetuned-libero-spatial'  # ⚠️ EDIT ME (a directory with 4-shard safetensors)
PROJ_CKPT   = REPO / 'models/pretrain_rlds_siglip_day8/proj_head.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = SiglipEncoderWrapper(OPENVLA_DIR).to(device).eval()
proj    = MultiViewProjectionHead(in_dim=1152, hidden=512, out_dim=128).to(device).eval()

ckpt = torch.load(PROJ_CKPT, map_location=device, weights_only=False)
proj.load_state_dict(ckpt['proj_head'])

print(f"encoder: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M params (frozen)")
print(f"proj_head: {sum(p.numel() for p in proj.parameters())/1e3:.1f}K params (loaded)")
print(f"trained on: {ckpt['config']['rlds_root']}")
print(f"epochs: {ckpt['config']['epochs']}, batch_size: {ckpt['config']['batch_size']}")

## 2. Pull 32 LIBERO frames (1 episode of each suite, 4 random anchor + 4 paired Δ=5 frames)

In [ ]:
from pretrain.dataset_rlds import LiberoRLDSPretrainDataset

RLDS_ROOT = '/path/to/modified_libero_rlds'   # ⚠️ EDIT ME

dataset = LiberoRLDSPretrainDataset(
    rlds_root=RLDS_ROOT,
    suites=('spatial', 'object', 'goal', '10'),
    delta_steps=5,
    max_per_episode=8,         # 8 anchors per episode
    max_episodes_per_suite=1,  # 1 episode per suite
    target_size=224,
)
print(f'{len(dataset)} samples')

## 3. Forward all frames → 128-d embeddings

In [ ]:
# Build batch: agent_t and wrist_t
images_a, images_w, instrs, suites, ts = [], [], [], [], []
for d, m in zip(dataset, [dataset.episodes[s['ep_idx']] for s in dataset.samples]):
    images_a.append(d['agent_t'])
    images_w.append(d['wrist_t'])
    instrs.append(m['instr'])
    suites.append(m['suite'])
    ts.append(d['episode_id'])    # actually ep_idx; t is in samples list

# Forward
with torch.no_grad():
    x_a = torch.stack(images_a).to(device)
    x_w = torch.stack(images_w).to(device)
    z_a = proj(encoder(x_a)).cpu()
    z_w = proj(encoder(x_w)).cpu()

print(f'agent embeddings: {tuple(z_a.shape)}')
print(f'wrist embeddings: {tuple(z_w.shape)}')
print(f'sample suites: {suites[:8]}')

## 4. Cosine similarity heatmap (agent vs agent)

In [ ]:
# Already L2-normed → cosine = dot
sim_aa = (z_a @ z_a.T).numpy()

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
im = ax.imshow(sim_aa, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Cosine similarity: agent_t × agent_t  (same-episode block should stand out)\n'
             'order: spatial(0..7) | object(8..15) | goal(16..23) | long10(24..31)')
ax.set_xlabel('frame index')
ax.set_ylabel('frame index')

# Draw block boundaries (every 8 frames = 1 episode)
for b in [8, 16, 24]:
    ax.axhline(b - 0.5, color='black', linewidth=1)
    ax.axvline(b - 0.5, color='black', linewidth=1)

plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(f'\nQuant check:')
print(f'  same-episode mean sim:  {sim_aa[:8, :8].mean():.3f}  (spatial block)')
print(f'  cross-task mean sim:    {sim_aa[:8, 8:].mean():.3f}')
print(f'  cross-task / same-task ratio: {sim_aa[:8, 8:].mean() / max(sim_aa[:8, :8].mean(), 1e-6):.2f}  (lower = better discrimination)')

## 5. Cross-modal: agent → wrist (same step)

Multi-view contrast was trained explicitly. Diagonal of `(z_agent_i · z_wrist_i)` should be **noticeably higher** than off-diagonal.

In [ ]:
sim_aw = (z_a @ z_w.T).numpy()

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
im = ax.imshow(sim_aw, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Cosine similarity: agent_t × wrist_t  (diagonal = same step)')
ax.set_xlabel('wrist frame index')
ax.set_ylabel('agent frame index')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

diag = np.diag(sim_aw)
off  = sim_aw - np.diag(np.diag(sim_aw))   # off-diagonal
print(f'\nCross-view (agent → wrist):')
print(f'  diagonal (same step) mean sim: {diag.mean():.3f}')
print(f'  off-diagonal mean sim:         {off.mean():.3f}')
print(f'  margin (diag - off):           {diag.mean() - off.mean():.3f}  (>0 means proj head learned cross-view alignment)')

## Summary

If you see:

- ✓ Block-diagonal pattern in the agent×agent heatmap (same-episode pairs > cross-task)
- ✓ Diagonal margin > 0.1 in the agent×wrist heatmap

…then the proj_head ckpt works as expected.

For quantitative recall@k metrics over **all 6000 frames**, see the sibling
`retrieval_eval.py` and the JSON / PNG it produces.